In [0]:
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import col

LOCATIONS_TABLE = "bronze_layer_reading_data_engine.bronze_layer_database.tb_locations"
TARGET_TABLE = "silver_layer_cleaning_data_engine.silver_layer_database.tb_beaches"

dictnames = {
    1: 'Manguinhos',
    2: 'Praia da Costa',
    3: 'Camburi',
    4: 'Carapebus',
    5: 'Jacaraípe',
    6: 'Bicanga',
    7: 'Nova Almeida',
    8:'Guarapari',
    9:'Itaparíca'
}

df = (
    spark.table(LOCATIONS_TABLE)
)

@dlt.table(
    name="tb_beaches",
    comment="Lista de Praias"
)
def tb_beaches():
    id = df.agg({"location_id": "max"}).collect()[0][0] + 1
    index = 0
    dictbeaches = []
    for row in df.collect():
        if(row['source_file'] == 'open-meteo-20.21S40.21W19m.csv'):
            beach_id = row['location_id'] + 1
            name = dictnames[beach_id]
            dictbeaches.append({'beach_id': beach_id, 'beach_name': name, 'latitude': row['latitude'], 'longitude': row['longitude'], 'elevation_m': row['elevation_m'], 'utc_offset_seconds': row['utc_offset_seconds'], 'timezone': row['timezone'], 'timezone_abbreviation': row['timezone_abbreviation'], 'file_id': row['location_id'], 'source_file': row['source_file']})
        if(row['source_file'] == 'open-meteo-20.21S40.21W28m.csv'):
            id += 1
            beach_id = id 
            name = dictnames[beach_id]
            dictbeaches.append({'beach_id': beach_id, 'beach_name': name, 'latitude': row['latitude'], 'longitude': row['longitude'], 'elevation_m': row['elevation_m'], 'utc_offset_seconds': row['utc_offset_seconds'], 'timezone': row['timezone'], 'timezone_abbreviation': row['timezone_abbreviation'], 'file_id': row['location_id'], 'source_file': row['source_file']})


    return spark.createDataFrame(dictbeaches)



In [0]:
RAW_TABLE = "bronze_layer_reading_data_engine.bronze_layer_database.tb_wave_raw_data"
BEACH_TABLE = "silver_layer_cleaning_data_engine.silver_layer_database.tb_beaches"
TARGET_TABLE = "silver_layer_cleaning_data_engine.silver_layer_database.tb_wave_data"

from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


dfBeach = (
    spark.table(BEACH_TABLE)
)
dfRawWave = (
    spark.table(RAW_TABLE)
)

@dlt.table(
    name="tb_wave_data_not_qualified",
    comment="Wave data com testes de qualidade não aplicados"
)
def tb_wave_data_not_qualified():
    dfUniqueId = dfRawWave.select('location_id', 'source_file').distinct()
    dfBeachIdLocationId = dfBeach.select('beach_id', 'file_id', 'source_file').distinct()
    dfBeachIdLocationId = dfBeachIdLocationId.join(dfUniqueId, dfBeachIdLocationId.file_id == dfUniqueId.location_id, 'inner').where(dfUniqueId.source_file == dfBeachIdLocationId.source_file).drop(dfUniqueId.location_id).drop(dfBeachIdLocationId.source_file)
    dictWave = []
    dfWaveData = dfRawWave.join(dfBeachIdLocationId, (dfBeachIdLocationId.file_id == dfRawWave.location_id) & (dfRawWave.source_file == dfBeachIdLocationId.source_file), 'inner').drop(dfBeachIdLocationId.source_file).drop(dfBeachIdLocationId.file_id)

    # Define a window specification
    window_spec = Window.orderBy('time')
    # Add a strictly sequential ID column
    df_with_seq_id = dfWaveData.withColumn("wave_id", row_number().over(window_spec))

    dfColumnRename = df_with_seq_id.withColumnRenamed("time", 'Forecast_Date').withColumnRenamed("wave_height_m", 'Wave_Height').withColumnRenamed("wave_direction_deg", 'Wave_Direction').withColumnRenamed("wave_period_s", 'Wave_Period').withColumnRenamed("wave_peak_period", 'Wave_Peak_Period').withColumnRenamed("source_file", 'Source_File').withColumnRenamed("beach_id", 'Beach_ID').withColumnRenamed("location_id", 'File_ID').withColumnRenamed("wave_id", 'Wave_ID')

    dfTypeCast = dfColumnRename.withColumn("Wave_Height", dfColumnRename["Wave_Height"].cast("double")).withColumn("Wave_Direction", dfColumnRename["Wave_Direction"].cast("double")).withColumn("Wave_Period", dfColumnRename["Wave_Period"].cast("double")).withColumn("Wave_Peak_Period", dfColumnRename["Wave_Peak_Period"].cast("double")).withColumn("Source_File", dfColumnRename["Source_File"].cast("string")).withColumn("Beach_ID", dfColumnRename["Beach_ID"].cast("long")).withColumn("File_ID", dfColumnRename["File_ID"].cast("long")).withColumn("Wave_ID", dfColumnRename["Wave_ID"].cast("long")).withColumn('Forecast_Date', dfColumnRename.Forecast_Date.cast("timestamp"))
    return dfTypeCast





In [0]:
import dlt
import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, TimestampType, DoubleType

# ── Configurações ──────────────────────────────────────────────────────────────

GRADIENT_LIMIT = 5

BOUNDS = {
    "height":    {"min": 0.5,  "max": 15},
    "direction": {"min": 0.01, "max": 360},
    "period":    {"min": 5,    "max": 10},
}

WAVE_DATA_SILVER_TABLE = "silver_layer_cleaning_data_engine.silver_layer_database.tb_wave_data"
BEACH_TABLE            = "silver_layer_cleaning_data_engine.silver_layer_database.tb_beaches"

# ── Funções de qualidade ───────────────────────────────────────────────────────

def spike_test(value, prev1, prev2, min_val, max_val):
    if prev1 is None:
        return 1
    avg = (value + prev1) / 2 if prev2 is None else (value + prev1 + prev2) / 3
    spike_ref = prev1 - avg
    if spike_ref > max_val or spike_ref <= min_val:
        return 0
    return 1

def dominio_test(value, min_val, max_val):
    return 0 if value < min_val or value > max_val else 1

def gradiente_test(value, prev1, limit):
    if prev1 is None:
        return 1
    return 0 if abs(value - prev1) > limit else 1

def wave_quality_test(value, prev1, prev2, metric_type):
    mn = BOUNDS[metric_type]["min"]
    mx = BOUNDS[metric_type]["max"]
    return (
        spike_test(value, prev1, prev2, mn, mx)
        * dominio_test(value, mn, mx)
        * gradiente_test(value, prev1, GRADIENT_LIMIT)
    )

# ── Tabela DLT ─────────────────────────────────────────────────────────────────

@dlt.table(
    name="tb_wave_data",
    comment="Wave data com testes de qualidade aplicados"
)
def tb_wave_data():
    df_wave = (
        spark.table(WAVE_DATA_SILVER_TABLE)
        .filter(F.col("Forecast_Date") >= "2026-09-26T00:00:00.000+00:00")
        .orderBy("Beach_ID", "Forecast_Date")
    )

    beach_ids = [
        r["Beach_ID"]
        for r in spark.table(BEACH_TABLE).select("Beach_ID").collect()
    ]

    schema = StructType([
        StructField("Beach_ID",       LongType(),      True),
        StructField("Forecast_Date",  TimestampType(), True),
        StructField("Wave_Height",    DoubleType(),    True),
        StructField("Wave_Direction", DoubleType(),    True),
        StructField("Wave_Period",    DoubleType(),    True),
        StructField("Aproved",        IntegerType(),   True),
    ])

    all_results = []

    for beach_id in beach_ids:
        rows = (
            df_wave
            .filter(F.col("Beach_ID") == beach_id)
            .orderBy("Forecast_Date")
            .collect()
        )

        approved_history = {"height": [], "direction": [], "period": []}

        for row in rows:
            h  = row["Wave_Height"]
            d  = row["Wave_Direction"]
            p  = row["Wave_Period"]
            dt = row["Forecast_Date"]

            def get_prev(metric, idx):
                hist = approved_history[metric]
                return hist[idx] if len(hist) > idx else None

            approved = (
                wave_quality_test(h, get_prev("height",    0), get_prev("height",    1), "height")
                * wave_quality_test(d, get_prev("direction", 0), get_prev("direction", 1), "direction")
                * wave_quality_test(p, get_prev("period",   0), get_prev("period",   1), "period")
            )

            all_results.append((beach_id, dt, h, d, p, int(approved)))

            if approved == 1:
                for metric, val in [("height", h), ("direction", d), ("period", p)]:
                    approved_history[metric].insert(0, val)
                    approved_history[metric] = approved_history[metric][:2]

    return spark.createDataFrame(all_results, schema=schema)

In [0]:
from pyspark.sql import functions as F, Row
import datetime
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType, IntegerType, BooleanType
from pyspark.sql.window import Window

TARGET_TABLE           = "gold_layer_data_engine.gold_layer_database.tb_wave_data"
BEACH_TABLE            = "silver_layer_cleaning_data_engine.silver_layer_database.tb_beaches"
SCALED_WAVE_DATA           = "gold_layer_data_engine.gold_layer_database.tb_wave_data_scaled"
AVG_WAVE_DATA           = "gold_layer_data_engine.gold_layer_database.tb_lower_avg_wave_height"

df_wave = (
    spark.table(TARGET_TABLE)
    .filter(F.col("Aproved") == 1)
    .orderBy("Beach_ID", "Forecast_Date")
)
df_beach = (spark.table(BEACH_TABLE))

df_wave_beach = df_wave.join(df_beach, on="Beach_ID", how="inner").drop(df_beach.beach_id).drop(df_beach.file_id).drop(df_beach.source_file).drop(df_beach.timezone).drop(df_beach.timezone_abbreviation).drop(df_beach.utc_offset_seconds)

DOUGLAS_SCALE = {
    "Calmo":    {"min": 0,  "max": 0},
    "Encrespado":    {"min": 0,  "max": 0.1},
    "Suave ":    {"min": 0.1,  "max": 0.5},
    "Fraco":    {"min": 0.5,  "max": 1.25},
    "Moderado":    {"min": 1.25,  "max": 2.5},
    "Grosso":    {"min": 2.5,  "max": 4.0},
    "Muito Grosso":    {"min": 4.0,  "max": 6.0},
    "Alto":    {"min": 6.0,  "max": 9.0},
    "Muito Alto":    {"min": 9.0,  "max": 14.0},
    "Fenomenal":    {"min": 15.0,  "max": 15.0},
}



def get_wave_scale(wave_height):
    for key, value in DOUGLAS_SCALE.items():
        if value["min"] <= wave_height <= value["max"]:
            return key
    return "Unknown"

"""Salva a tabela com os dados de onda aprovados e com a escala de Douglas"""
@dlt.table(
    name="tb_wave_data_scaled",
    comment="Wave data com testes de qualidade aplicados e categorizado com a escala de Douglas"
)
def tb_wave_data_scaled():
    df_wave_beach_scaled = df_wave_beach.withColumn("Wave_Scale", F.udf(get_wave_scale, returnType=StringType())(F.col("Wave_Height")))
    return df_wave_beach_scaled

"""Salva a tabela com qual praia tem a menor onda por dia"""
@dlt.table(
    name="tb_lower_avg_wave_height",
    comment="Tabela com qual praia é a mais calma por dia."
)

def tb_lower_avg_wave_height():
    df_wave_beach_date = df_wave_beach.withColumn("Forecast_Date", F.to_timestamp(F.col("Forecast_Date")))
    DATA_INICIO = "2026-09-26"
    DATA_FIM    = "2026-09-29"

    df_gold = (
        df_wave_beach_date
        .filter(F.col("Forecast_Date").between(DATA_INICIO, DATA_FIM))
    )

    df_com_dia = df_gold.withColumn("Dia", F.to_date("Forecast_Date"))

    df_media = (
        df_com_dia
        .groupBy("Dia", "Beach_ID")
        .agg(F.avg("Wave_Height").alias("Media_Altura"))
    )

    w = Window.partitionBy("Dia").orderBy("Media_Altura")

    df_menor_por_dia = (
        df_media
        .withColumn("rank", F.row_number().over(w))
        .filter(F.col("rank") == 1)
        .drop("rank")
    )

    dict_result = []

    df_menor_altura = df_menor_por_dia.orderBy("Media_Altura")

    for row in df_menor_por_dia.collect():
        beach_name = df_beach.filter(F.col("beach_id") == row["Beach_ID"]).select("beach_name").collect()[0][0]
        scale = get_wave_scale(row["Media_Altura"])
        dict_result.append({
            "Beach_Name": beach_name,
            "Dia": row["Dia"],
            "Ondas":scale
        })
    

    df_result = spark.createDataFrame(dict_result)
    f_result = df_result.withColumn("Dia", F.to_date(F.col("Dia")))
    df_result = df_result.withColumn("Dia", F.date_format(F.col("Dia"), "dd/MM/yyyy"))

    return df_result




